## Data loader workflow — full corpus
Lists every raw AI4Arctic scene under the S3 prefix, downloads each to a temp file, and runs it through `training.load_scene`/`yield_chips` (B2.1), tallying chip counts and mean valid fraction per scene.

This is the *loading* half of the eventual B2.6 assembly harness — it doesn't call B2.2–B2.5 (encoding, patch features, labels, assembly) and writes nothing to S3. It exists to prove the B2.1 loader holds up across the whole corpus, not just the one scene used in `notebooks/building_training_dataset_single_img/`. Each scene is downloaded, processed, and deleted one at a time, so disk usage stays bounded regardless of corpus size; one scene failing is recorded and skipped rather than stopping the run.

In [ ]:
import tempfile
import time
from pathlib import Path

import boto3
from mypy_boto3_s3 import S3Client

from training import ALL_BANDS, load_band_means, load_scene, yield_chips

### Configuration
`MAX_SCENES` caps how many scenes to process -- set to `None` to run the whole corpus. Useful to leave at a small number the first time, given each scene is a multi-hundred-MB download.

In [ ]:
BUCKET = "prescient-ice-data"
S3_PREFIX = "training_data/ai4arctic/raw_train/"
STATS_KEY = "training_data/ai4arctic/statistics/dataset_stats.json"
AWS_PROFILE = "spk_data"

MAX_SCENES = 5  # None to process the full corpus

### List all raw scene keys in S3

In [ ]:
def list_scene_keys(s3: S3Client, bucket: str, prefix: str) -> list[str]:
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".nc"):
                keys.append(obj["Key"])
    return keys


session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3")

scene_keys = list_scene_keys(s3, BUCKET, S3_PREFIX)
if MAX_SCENES is not None:
    scene_keys = scene_keys[:MAX_SCENES]

print(f"Scenes to process: {len(scene_keys)}")

### Load per-band normalisation stats (B1)

In [ ]:
band_means = load_band_means(BUCKET, STATS_KEY, ALL_BANDS, profile=AWS_PROFILE)
print(f"Loaded stats for {len(band_means)} bands")

### Load every scene through the B2.1 loader
For each scene: download to a temp file, call `load_scene` (opens the NetCDF once, builds the valid mask, substitutes, resamples ancillary, builds GCP interpolators, parses CT labels), then `yield_chips` to count chips and average valid fraction. Chips themselves are never materialised beyond this per-scene loop -- only the summary is kept, matching B2.1's single-pass, no-intermediate-store design.

In [ ]:
def summarize_scene(s3: S3Client, bucket: str, key: str, band_means: dict[str, float]) -> dict:
    with tempfile.TemporaryDirectory() as tmpdir:
        local_path = Path(tmpdir) / key.split("/")[-1]
        s3.download_file(bucket, key, str(local_path))

        scene = load_scene(local_path, band_means)
        n_chips = 0
        valid_fraction_total = 0.0
        for chip in yield_chips(scene):
            n_chips += 1
            valid_fraction_total += float(chip.valid_mask.mean())

    return {
        "scene_id": scene.scene_id,
        "n_chips": n_chips,
        "mean_valid_fraction": valid_fraction_total / n_chips if n_chips else float("nan"),
    }

In [ ]:
results = []
errors = []
t0 = time.time()

for i, key in enumerate(scene_keys, start=1):
    try:
        summary = summarize_scene(s3, BUCKET, key, band_means)
        results.append(summary)
        print(
            f"[{i}/{len(scene_keys)}] {summary['scene_id']}: "
            f"{summary['n_chips']} chips, "
            f"mean valid_fraction={summary['mean_valid_fraction']:.3f}"
        )
    except Exception as e:
        errors.append((key, str(e)))
        print(f"[{i}/{len(scene_keys)}] ERROR {key}: {e}")

elapsed = time.time() - t0
print(f"\nProcessed {len(results)}/{len(scene_keys)} scenes in {elapsed:.1f}s")
print(f"Total chips: {sum(r['n_chips'] for r in results)}")
if errors:
    print(f"Errors: {len(errors)}")
    for key, msg in errors:
        print(f"  {key}: {msg}")